In [1]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import softmax
from itertools import product

import utilities as utils

In [2]:
save_data_flag = True

## Bead prediction experiment

In [14]:
beads_datadir = "./Data/beads/"
with open(beads_datadir + 'flipped_sjs.txt', 'r',  encoding='utf-16') as f:
    flipped_sjs = [int(line.strip()) for line in f.readlines()]
beads_trial_seq = pd.read_csv(beads_datadir + "trial_sequence.csv")
preprocdf = pd.read_csv(beads_datadir + "sj-preproc-data.csv")

wsize = 7 # number of observed beads to include in the "window" used to compute choice probabilities and posterior probabilities for each unique X (sequence of observed beads of length wsize, from current trial backwards)

# generative parameters for the bead-prediction experiment
h_=0.99 # jar stay probability
p0_=0.8 # probability of drawing bead type 0 from jar 0
p1_=0.2 # probability of drawing bead type 0 from jar 1
P0 = np.array([[0.5,0.5]]) # prior over jars for computing posterior probabilities over hidden markov process
H = np.ones((2,2)) - np.abs(np.eye(2)*-1 + h_) # transition matrix
E = np.vstack((np.array([[1,0]])*p0_ + np.array([[0,1]])*(1-p0_),np.array([[1,0]])*p1_ + np.array([[0,1]])*(1-p1_))) # emission matrix

modeldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'run': [],
    'opt_betastar': [],
    'opt_loss': [],
    'opt_BIC': [],
    'oneback_betastar': [],
    'oneback_loss': [],
    'oneback_BIC': [],
    'fullparam_betastar': [],
    'fullparam_h': [],
    'fullparam_loss': [],
    'fullparam_BIC': [],
    'best_model': [],
    'best_model_betastar': [],
    'best_model_loss': [],
    'best_model_BIC': []
})

beads_emp, jars_emp = utils.getXY_beads(beads_trial_seq['bead'].to_numpy(), beads_trial_seq['jar'].to_numpy(), wsize)
beads_emp1b, jars_emp1b = utils.getXY_beads(beads_trial_seq['bead'].to_numpy(), beads_trial_seq['jar'].to_numpy(), wsize=1,ref_wsize=wsize)
p_YgX_opt = utils.P_jar_g_beads(beads_emp,E,H,P0)
p_YgX_1b = utils.P_jar_g_beads(beads_emp1b,E,H,P0)

Ntrials = beads_emp.shape[0]

fully_opt_model = lambda params: softmax(params[0]*p_YgX_opt, axis=1)
oneback_model = lambda params: softmax(params[0]*p_YgX_1b, axis=1)

def fullparam_model(params):
    betastar, h = params
    H_ = np.ones((2,2)) - np.abs(np.eye(2)*-1 + h) # transition matrix
    p_YgX_fullparam = utils.P_jar_g_beads(beads_emp,E,H_,P0)
    return softmax(betastar*p_YgX_fullparam, axis=1)

def fit_fullparam_model(obj_fn):
    bounds = [(0.0, None),(0.0, 1.0)]
    best_result = None
    best_loss = np.inf
    betastar0 = np.array([0.1, 1., 3., 5., 8.])
    h0 = np.array([0.6, 0.7, 0.8, 0.9, 0.99])
    inds = product(range(len(betastar0)),range(len(h0)))
    for ind in inds:
        X0 = np.array([betastar0[ind[0]], h0[ind[1]]])
        result = minimize(obj_fn,X0,bounds=bounds,options={'eps': 1e-19})
        if result.fun <= best_loss:
            best_result = result
            best_loss = result.fun
    if best_result is None:
        return result
    else:
        return best_result

for sj in preprocdf['subject_index'].unique():
    for run in [1,2]:
        df_ = preprocdf[(preprocdf['subject_index']==sj) & (preprocdf['run']==run)]

        sjdf = pd.DataFrame({
            'subject_ID': [df_['subject_ID'].iloc[0]],
            'subject_index': [sj],
            'run': [run],
        })

        choices = utils.getR_beads(df_['choice'].to_numpy(),wsize)
        if sj in flipped_sjs:
            choices = 1 - choices

        oneback_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,oneback_model,params)
        result = minimize(oneback_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['oneback_betastar'] = result.x[0]
        sjdf['oneback_loss'] = result.fun
        sjdf['oneback_BIC'] = utils.compute_BIC(1, Ntrials, result.fun) # 1 free parameter (betastar) for this model
        sjdf['best_model'] = 'oneback'
        sjdf['best_model_betastar'] = result.x[0]
        sjdf['best_model_loss'] = result.fun
        sjdf['best_model_BIC'] = sjdf['oneback_BIC'].iloc[0]
        
        fully_opt_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fully_opt_model,params)
        result = minimize(fully_opt_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['opt_betastar'] = result.x[0]
        sjdf['opt_loss'] = result.fun
        sjdf['opt_BIC'] = utils.compute_BIC(1, Ntrials, result.fun) # 1 free parameter (betastar) for this model
        if sjdf['opt_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'fully_optimal'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['opt_BIC'].iloc[0]

        fullparam_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fullparam_model,params)
        result = fit_fullparam_model(fullparam_obj)
        sjdf['fullparam_betastar'] = result.x[0]
        sjdf['fullparam_h'] = result.x[1]
        sjdf['fullparam_loss'] = result.fun
        sjdf['fullparam_BIC'] = utils.compute_BIC(2, Ntrials, result.fun) # 2 free parameters (betastar, h) for this model
        if sjdf['fullparam_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'fullparam'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['fullparam_BIC'].iloc[0]

        modeldf = pd.concat((modeldf, sjdf), ignore_index=True)

    print(f'Subject {sj} done')

if save_data_flag:
    modeldf.to_csv(beads_datadir + 'model_fits.csv', index=False)


Subject 0.0 done
Subject 1.0 done
Subject 2.0 done
Subject 3.0 done
Subject 4.0 done
Subject 5.0 done
Subject 6.0 done
Subject 7.0 done
Subject 8.0 done
Subject 9.0 done
Subject 10.0 done
Subject 11.0 done
Subject 12.0 done
Subject 13.0 done
Subject 14.0 done
Subject 15.0 done
Subject 16.0 done
Subject 17.0 done
Subject 18.0 done
Subject 19.0 done
Subject 20.0 done
Subject 21.0 done
Subject 22.0 done
Subject 23.0 done
Subject 24.0 done
Subject 25.0 done
Subject 26.0 done


/workspaces/human-inference-IB/utilities.py:688: RuntimeWarning: divide by zero encountered in log
  return -np.nansum(choices_oh*np.log(model_probs))
/workspaces/human-inference-IB/utilities.py:688: RuntimeWarning: invalid value encountered in multiply
  return -np.nansum(choices_oh*np.log(model_probs))
/usr/local/lib/python3.11/site-packages/scipy/optimize/_numdiff.py:576: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0


Subject 27.0 done
Subject 28.0 done
Subject 29.0 done
Subject 30.0 done
Subject 31.0 done
Subject 32.0 done
Subject 33.0 done
Subject 34.0 done
Subject 35.0 done
Subject 36.0 done
Subject 37.0 done
Subject 38.0 done
Subject 39.0 done
Subject 40.0 done
Subject 41.0 done
Subject 42.0 done
Subject 43.0 done
Subject 44.0 done
Subject 45.0 done
Subject 46.0 done
Subject 47.0 done
Subject 48.0 done
Subject 49.0 done
Subject 50.0 done
Subject 51.0 done
Subject 52.0 done
Subject 53.0 done
Subject 54.0 done
Subject 55.0 done
Subject 56.0 done
Subject 57.0 done
Subject 58.0 done
Subject 59.0 done
Subject 60.0 done
Subject 61.0 done
Subject 62.0 done
Subject 63.0 done
Subject 64.0 done
Subject 65.0 done
Subject 66.0 done
Subject 67.0 done
Subject 68.0 done
Subject 69.0 done


In [15]:
modeldf

,subject_ID,subject_index,run,opt_betastar,opt_loss,opt_BIC,oneback_betastar,oneback_loss,oneback_BIC,fullparam_betastar,fullparam_h,fullparam_loss,fullparam_BIC,best_model,best_model_betastar,best_model_loss,best_model_BIC
0,6107f24d21eaa93618cf9191,0.0,1.0,2.410763,166.080933,338.362376,2.098824,260.447363,527.095236,2.824846,0.958019,156.955288,326.311593,fullparam,2.824846,156.955288,326.311593
1,6107f24d21eaa93618cf9191,0.0,2.0,3.972164,70.802576,147.805662,2.059817,262.942498,532.085505,4.018743,0.988208,70.706887,153.814793,fully_optimal,3.972164,70.802576,147.805662
2,5f4d296913fe2f98d46f520c,1.0,1.0,1.008264,295.337193,596.874895,1.908554,272.463659,551.127827,1.832767,0.691215,268.742220,549.885458,fullparam,1.832767,268.742220,549.885458
3,5f4d296913fe2f98d46f520c,1.0,2.0,0.353340,335.457629,677.115767,0.469614,336.877070,679.954650,0.458617,0.901089,334.298442,680.997903,fully_optimal,0.353340,335.457629,677.115767
4,6441a242a7d85d5c3e49a173,2.0,1.0,4.453653,54.179084,114.558677,2.002230,266.598178,539.396865,4.406661,0.991588,54.078240,120.557497,fully_optimal,4.453653,54.179084,114.558677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,6501f64fbe9a0a65bd5f7891,67.0,2.0,4.857825,43.490631,93.181771,2.079257,261.700782,529.602073,5.004880,0.986647,43.114615,98.630249,fully_optimal,4.857825,43.490631,93.181771
136,5c55b95b91c0ad0001cf86ab,68.0,1.0,4.449063,54.316129,114.832767,2.098824,260.447363,527.095236,4.504367,0.988335,54.227671,120.856360,fully_optimal,4.449063,54.316129,114.832767
137,5c55b95b91c0ad0001cf86ab,68.0,2.0,4.768508,45.631601,97.463712,2.198647,254.002005,514.204518,4.949224,0.985751,45.109200,102.619419,fully_optimal,4.768508,45.631601,97.463712
138,663a5538ea4ded3dd872258e,69.0,1.0,1.656379,235.585572,477.371653,2.178408,255.315114,516.830738,2.168707,0.919277,220.790623,453.982264,fullparam,2.168707,220.790623,453.982264
